# Experiment: CTA Trend Walk-Forward Research Protocol


**Objective:** reserve an untouched final period and create reviewable chronological folds before any parameter search. This notebook does not rank parameters or inspect final-holdout performance.

**Locked hypothesis:** the existing long-only CTA Trend rules can provide repeatable net risk-adjusted benefit over a constant-exposure benchmark across a preregistered broad-ETF universe. Full acceptance gates are in `docs/research-protocol.md`.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import sys
from pathlib import Path

SEED = 17_291
ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    raise RuntimeError("Run this notebook from the repository root")
sys.path.insert(0, str(ROOT / "backend"))

from app.research import fold_manifest, reserve_final_holdout, walk_forward_folds
from app.store import load_bars


## Plan

1. Load local SPY bars only to prove the partition machinery. No network calls.
2. Reserve the latest 504 bars; expose only reservation metadata.
3. Build expanding 756/252/252 train/validation/test folds on development history.
4. Stop before parameter ranking. The grid, attempt ledger, and multiple-testing method must be locked first.

Planned primary metric: median net out-of-sample excess return over the constant-exposure benchmark. Planned gates: positive excess in at least 60% of folds, positive median Calmar, pooled drawdown no worse than -25%, and at least 30 closed out-of-sample trades.


In [ ]:
# Partition configuration is fixed before performance is calculated.
SYMBOL = "SPY"
HOLDOUT_BARS = 504
TRAIN_BARS = 756
VALIDATION_BARS = 252
TEST_BARS = 252

bars = load_bars(SYMBOL)
if bars.empty:
    raise RuntimeError("SPY is missing; run the documented local fetch first")
development, holdout = reserve_final_holdout(bars, holdout_bars=HOLDOUT_BARS)
del bars  # reduce the chance of casually inspecting the reserved values
folds = walk_forward_folds(
    development,
    train_bars=TRAIN_BARS,
    validation_bars=VALIDATION_BARS,
    test_bars=TEST_BARS,
)
{"development_bars": len(development), "folds": len(folds), "reserved_holdout": holdout}


## Results

The table below is a boundary manifest, not a performance result. Every row must satisfy `train_end < validation_start` and `validation_end < test_start`. The reserved final period must not appear in any row.


In [ ]:
manifest = fold_manifest(folds)
assert (manifest.train_end < manifest.validation_start).all()
assert (manifest.validation_end < manifest.test_start).all()
assert (manifest.test_end < holdout.start).all()
manifest


## Next steps

- Define and freeze the broad-ETF universe and missing-history policy.
- Add an append-only attempt ledger and the complete finite CTA parameter grid.
- Choose the multiple-comparison treatment before ranking any configuration.
- Implement fold-local selection with costs and parameter-stability output.
- Only after a model commit is locked, add a separate explicit final-holdout unlock step.
